# Reader 1.0
## FineWeb Dataset Slice Reader

This notebook loads and inspects one parquet slice from the FineWeb dataset.

### Goals
- Load one parquet slice from the dataset
- Inspect dataset structure
- Preview sample rows
- Check language and text statistics
- Produce basic summary information for downstream filtering

In [1]:
import os
import pandas as pd

## Step 1: Define a reusable dataset reader

This function loads a parquet file from the FineWeb data directory.
If no file name is provided, it loads the first parquet file in the directory.

In [2]:
import os
import pandas as pd

def load_parquet_file(base_dir, file_name=None):
    if not os.path.exists(base_dir):
        raise FileNotFoundError(f"Base directory does not exist: {base_dir}")

    parquet_files = sorted([
        f for f in os.listdir(base_dir)
        if f.endswith(".parquet")
    ])

    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found in {base_dir}")

    if file_name is None:
        file_name = parquet_files[0]

    file_path = os.path.join(base_dir, file_name)

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Parquet file does not exist: {file_path}")

    print("Loading file:", file_path)

    df = pd.read_parquet(file_path)
    return df, file_path, parquet_files

## Step 2: List available parquet files

This helps make the reader reusable for different parquet files.

In [3]:
base_dir = "/home/jovyan/data/fineweb"

parquet_files = sorted([
    f for f in os.listdir(base_dir)
    if f.endswith(".parquet")
])

print("Available parquet files:")
for f in parquet_files[:20]:
    print("-", f)

Available parquet files:
- 004_00018.parquet
- 004_00019.parquet
- 004_00020.parquet
- 004_00021.parquet
- 004_00022.parquet
- 004_00023.parquet
- 004_00046.parquet
- 004_00047.parquet
- 004_00048.parquet
- 004_00049.parquet


## Step 3: Select a parquet file and load data

Select one slice file

In [4]:
base_dir = "/home/jovyan/data/fineweb"
file_name = None   # use the first parquet file by default

df, file_path, parquet_files = load_parquet_file(base_dir, file_name)

print("Loaded file:", file_path)
print("Shape:", df.shape)

Loading file: /home/jovyan/data/fineweb/004_00018.parquet
Loaded file: /home/jovyan/data/fineweb/004_00018.parquet
Shape: (172447, 9)


## Step 4: Show basic dataset information

In [6]:
print("Current working directory:", os.getcwd())
print("Base directory:", base_dir)
print("Loaded file:", os.path.basename(file_path))
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Current working directory: /home/jovyan
Base directory: /home/jovyan/data/fineweb
Loaded file: 004_00018.parquet
Dataset shape: (172447, 9)
Columns: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count']


In [7]:
df.head()

,text,id,dump,url,date,file_path,language,language_score,token_count
0,Welcome to Harrisonburg TownSquare.\nTownSquar...,<urn:uuid:189514cf-784a-4e9f-9e8e-f350c5a41adf>,CC-MAIN-2025-26,https://www.townsquareapp.co/,2025-06-16T16:22:40Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.899284,203
1,"The Art of Effective Board Reporting\nSep 5, 2...",<urn:uuid:9d2f701d-48f5-418f-a8e2-d5b2d51aad36>,CC-MAIN-2025-26,https://www.traact.com/blog/the-art-of-effecti...,2025-06-16T16:45:14Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.942698,2717
2,Belinda's journey and transformation is a real...,<urn:uuid:ccb16b4e-f301-480c-9231-33f6e823eabe>,CC-MAIN-2025-26,https://www.tracydixonmindandbodyfit.com/post/...,2025-06-16T15:44:36Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.987144,555
3,Many people purchase plain shirts or other clo...,<urn:uuid:42451264-d511-45ac-8afb-d27558b16a33>,CC-MAIN-2025-26,https://www.transportdirectory.org/items-that-...,2025-06-16T16:20:43Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.976207,469
4,Beijing Itinerary 5 Days: What to See & How to...,<urn:uuid:b3df7fbd-18f6-4bab-b074-969a3f6425c5>,CC-MAIN-2025-26,https://www.travelchinaguide.com/package/5-day...,2025-06-16T16:09:17Z,s3://commoncrawl/crawl-data/CC-MAIN-2025-26/se...,en,0.946699,1496


## Step 6: Inspect data types

In [8]:
df.dtypes

text                  str
id                    str
dump                  str
url                   str
date                  str
file_path             str
language              str
language_score    float64
token_count         int64
dtype: object

## Step 7: Check missing values

In [9]:
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_summary

text              0
id                0
dump              0
url               0
date              0
file_path         0
language          0
language_score    0
token_count       0
dtype: int64

## Step 8: Check language distribution

In [10]:
df["language"].value_counts().head(20)

language
en    172447
Name: count, dtype: int64

## Step 9: Summarize language confidence

In [11]:
df["language_score"].describe()


count    172447.000000
mean          0.935297
std           0.039921
min           0.650054
25%           0.918842
50%           0.943012
75%           0.962026
max           0.998501
Name: language_score, dtype: float64

## Step 10: Analyze text length

In [12]:
df["text_length"] = df["text"].str.len()
df["text_length"].describe()

count    172447.000000
mean       3638.066304
std        5759.373853
min         136.000000
25%        1111.000000
50%        2262.000000
75%        4286.000000
max      481614.000000
Name: text_length, dtype: float64

## Step 11: Estimate token count

This is a rough estimated token count based on whitespace splitting.
It is not the same as model tokenizer output, but it is useful for basic filtering.

In [17]:
df["estimated_token_count"] = df["text"].str.split().str.len()
df["estimated_token_count"].describe()

count    172447.000000
mean        594.024286
std         952.315807
min          22.000000
25%         181.000000
50%         370.000000
75%         699.000000
max       76905.000000
Name: estimated_token_count, dtype: float64

## Step 12: Inspect shortest samples

In [18]:
df[["text", "url", "language_score", "text_length", "estimated_token_count"]] \
    .sort_values("text_length") \
    .head(10)

,text,url,language_score,text_length,estimated_token_count
138234,top of page\nfood & drink\nUse tab to navigate...,https://www.picksbar.com/media,0.767501,136,25
77829,MPC Modern Pioneer\nWHO WE ARE\nConnecting The...,https://www.mpcpro.com/,0.655760,137,26
135970,top of page\nHome & Living\nUse tab to navigat...,https://www.btwchurch.com/product-page/unisex-...,0.657335,140,28
146797,top of page\nUse tab to navigate through the m...,https://www.gemrain.net/download-outlines/sa-e...,0.721844,140,23
9522,Skip to main content\nRoad to Google Developer...,https://gdg.community.dev/forum/how-to-get-int...,0.856314,141,24
62722,Lindsey Vonn's Vegan Chili Recipe\nThis Recipe...,https://www.beyondmeat.com/en-US/recipes/linds...,0.874195,148,23
157213,Filter by type:\nEducare all'arte e alle emozi...,https://radiopapesse.org/en/archive/interviews...,0.763067,150,25
116951,Game Server List\nLogin or register\nDer LS Tr...,https://der-ls-treffpunkt.de/user/3613-nikoholic/,0.745134,150,27
77038,Work With Nick\nGet in touch with Nick Nanton ...,https://nicknanton.com/events/filming-its-happ...,0.768271,151,25
121252,top of page\nUse tab to navigate through the m...,https://www.2xexperience.com/shop?page=2,0.804102,152,26


In [19]:
df[["text", "url", "language_score", "text_length", "estimated_token_count"]] \
    .sort_values("text_length", ascending=False) \
    .head(10)

,text,url,language_score,text_length,estimated_token_count
84229,The pilot test (n=16) reported large amounts o...,https://sms201995.com/2024/09/,0.935425,481614,69204
32060,Australia a treasure-trove of literature treas...,https://www.gutenberg.net.au/ebooks01/0100151h...,0.989143,419700,76905
102144,and the Saints\nLet those who suffer according...,https://goingtojesus.com/suffering-and-the-sai...,0.977453,408132,73015
70135,Theosophy Home Our Facebook Group |\nA Study i...,https://www.anandgholap.net/Nirvana-GSA.htm,0.966035,353499,62591
45136,ON THE INTERIOR\nDES AFFAIRES INTÉRIEURES\nMon...,https://www.ola.org/en/legislative-business/co...,0.959962,300698,50327
51043,Case Number: HT 02 354\nNeutral Citation Numbe...,http://www.adjudication.co.uk/archive/iframe/c...,0.979010,279416,48358
12141,The Project Gutenberg EBook of Grace Harlowe's...,https://www.gutenberg.org/files/20341/20341-h/...,0.983838,273978,49300
90001,Solo Travel Fear #1: What about eating in a re...,https://whitby-taxis.co.uk/category/business/,0.836262,269808,47458
68602,First published in 1912.\nThis online edition ...,https://www.globalgreyebooks.com/online-ebooks...,0.964299,252873,42017
905,CONTENTS Preface Chapter 1 PORTRAITS OF THE AR...,https://pdfcoffee.com/2pac-vs-biggie-an-illust...,0.947122,246054,41350


## Step 14: Generate dataset summary

This summary can be reused for filtering design, reporting, or further analysis.

In [16]:
summary = {
    "file_path": file_path,
    "shape": df.shape,
    "columns": df.columns.tolist(),
    "missing_values": df.isnull().sum().to_dict(),
    "language_distribution": df["language"].value_counts().head(10).to_dict(),
    "language_score_summary": df["language_score"].describe().to_dict(),
    "text_length_summary": df["text_length"].describe().to_dict(),
    "token_count_summary": df["token_count"].describe().to_dict(),
}

summary

{'file_path': '/home/jovyan/data/fineweb/004_00018.parquet',
 'shape': (172447, 10),
 'columns': ['text',
  'id',
  'dump',
  'url',
  'date',
  'file_path',
  'language',
  'language_score',
  'token_count',
  'text_length'],
 'missing_values': {'text': 0,
  'id': 0,
  'dump': 0,
  'url': 0,
  'date': 0,
  'file_path': 0,
  'language': 0,
  'language_score': 0,
  'token_count': 0,
  'text_length': 0},
 'language_distribution': {'en': 172447},
 'language_score_summary': {'count': 172447.0,
  'mean': 0.9352966696017893,
  'std': 0.039920747330456544,
  'min': 0.6500535607337952,
  '25%': 0.9188422560691833,
  '50%': 0.9430122375488281,
  '75%': 0.9620256423950195,
  'max': 0.9985013008117676},
 'text_length_summary': {'count': 172447.0,
  'mean': 3638.0663044297667,
  'std': 5759.373853036025,
  'min': 136.0,
  '25%': 1111.0,
  '50%': 2262.0,
  '75%': 4286.0,
  'max': 481614.0},
 'token_count_summary': {'count': 172447.0,
  'mean': 594.0242857225699,
  'std': 952.3158068979558,
  'min':

## Initial Observations

- The dataset was successfully loaded from a parquet file.
- Each row appears to represent one web text sample with metadata.
- Key fields include text, id, dump, url, date, file_path, language, language_score, and token_count.
- This notebook is intended as Reader 1.0 for downstream cleaning and filtering work.

## Potential Cleaning Directions

- Keep English-only samples
- Filter out very short text
- Investigate low language_score rows
- Check duplicates in text and url
- Inspect low-information or template-like pages




## Initial Cleaning Trial

This section tests a few simple filtering ideas based on the initial dataset inspection.

The goal here is not to build a complete cleaning pipeline yet, but to explore whether some basic filtering rules may be useful.

In [21]:
##remove non-english 
english_df = df[df["language"] == "en"]
print("Original shape:", df.shape)
print("English-only shape:", english_df.shape)

Original shape: (172447, 11)
English-only shape: (172447, 11)


In [22]:
##keep high-level language confidence
high_conf_df = english_df[english_df["language_score"] >= 0.9]
print("English-only shape:", english_df.shape)
print("English + high language_score shape:", high_conf_df.shape)

English-only shape: (172447, 11)
English + high language_score shape: (148399, 11)


In [23]:
##remove short texts
length_filtered_df = high_conf_df[high_conf_df["text_length"] >= 200]
print("Before text length filtering:", high_conf_df.shape)
print("After text length filtering:", length_filtered_df.shape)

Before text length filtering: (148399, 11)
After text length filtering: (148391, 11)


In [24]:
##remove duplicated URL
duplicate_url_count = length_filtered_df["url"].duplicated().sum()
print("Number of duplicated URLs:", duplicate_url_count)
dedup_url_df = length_filtered_df.drop_duplicates(subset=["url"])
print("Before URL deduplication:", length_filtered_df.shape)
print("After URL deduplication:", dedup_url_df.shape)

Number of duplicated URLs: 13
Before URL deduplication: (148391, 11)
After URL deduplication: (148378, 11)


In [25]:
##remove duplicated text
duplicate_text_count = length_filtered_df["text"].duplicated().sum()
print("Number of duplicated text samples:", duplicate_text_count)
dedup_text_df = length_filtered_df.drop_duplicates(subset=["text"])
print("Before text deduplication:", length_filtered_df.shape)
print("After text deduplication:", dedup_text_df.shape)

Number of duplicated text samples: 0
Before text deduplication: (148391, 11)
After text deduplication: (148391, 11)


## My Current Thoughts

My current thinking is that, since the data we need to process all comes from websites, the content can roughly be divided into two broad categories.

The first category is information-oriented websites, such as the ANU official website, Wikipedia, or CNKI (a Chinese academic search platform).  
The second category is forum-style websites, such as Reddit or Japanese discussion forums like 2CH.

For the first category, I think there are some general filtering rules that can be applied across many websites. For example:
- remove empty text
- remove duplicated text
- remove short texts
- remove low-information content such as “main menu” or “disclaimer”
- remove duplicated or suspicious URLs
- remove advertisement links
- keep high-level language confidence

There are likely many other general rules of this kind.

At the same time, I think some filtering rules need to be more task-specific. For example, if the final goal is to find which government organisations in Canberra are currently hiring, then we should focus on recent official government websites rather than pages from older years such as 2018 or 2019.

So, in my view, general filtering rules can be designed at an early stage, but more detailed filtering decisions should depend on the final task requirements.